<a href="https://colab.research.google.com/github/syedadilejazz/Complete_GenAI_Series_Bappy_euron/blob/main/HOW_TO_RUN_LLAMA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#LLAMA 2

The Llama 2 is a collection of pretrained and fine-tuned generative text models, ranging from 7 billions to 70 billions parameters, designed for dialogue use cases.

It Outperforms open-source chat models on most benchmarks and is on par with popular closed-soure models in human evaluation for helpfulness and safety.

#https://huggingface.co/meta-llama/Llama-2-13b-chat

llama.cpp's objective is to run the LLAMA models with 4-bit integer quantization in MacBook. It is a plain C/C++ implementation optimized for Apple silicon and x86 architectures, supporting various integer quanatization and BLAS libraries. Orginally a web chat example, it now serves as a development playground for ggml library features.

GGML, a C library for machine learning, facilities the distribution of large language models(LLMs). It utilizes quantization to enable efficient LLM execution on consumer hardware. GGML files contan binary-encoded data, including version number, hyperparameters, vocabulary, and weights. The vocabulary comparies tokens for language generation, while the weights determine the LLM's size. Qunatization reduces preecision to optimize resource usage.

#Qunatized Models from the Hugging face Community

The Hugging Face community provides quantized models, which allow us to efficiently and effectively utlize the model on the T4 GPU. It is important to consult reliable source before using the model.

There are several variations available, but the ones that interest us are based on the GGLM Libraries.

We can see the difference varaitions that LLAMA-2_13B_GGML has

In this case, we will use the models caleed https://huggingface.co/TheBloke/Llama-2-13B-chat-GGML

In [ ]:
#Install All th Required Packages

In [ ]:
!nvidia-smi

Tue Sep  1 10:01:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA-CUBLAS=on" Force_CMAKE=1 pip install llama-cpp-python==0.1.78 numpy==1.23.4 --force-reinstall --upgrade --no-cache-dir --verbose
!pip install huggingface_hub
!pip install llama-cpp-python==0.1.78
!pip install numpy==1.23.4

Streaming output truncated to the last 5000 lines.
    Found link https://files.pythonhosted.org/packages/f9/59/701df637517d6af0434cbb580bfc35a9c536aa7f47e0c2e222f1ef83547c/setuptools-69.0.1-py3-none-any.whl (from https://pypi.org/simple/setuptools/) (requires-python:>=3.8), version: 69.0.1
    Found link https://files.pythonhosted.org/packages/fb/d0/d5744a7190a984ab728c7d2bd7e39bae2ff523538b233246fd34e2148566/setuptools-69.0.1.tar.gz (from https://pypi.org/simple/setuptools/) (requires-python:>=3.8), version: 69.0.1
    Found link https://files.pythonhosted.org/packages/bb/e1/ed2dd0850446b8697ad28d118df885ad04140c64ace06c4bd559f7c8a94f/setuptools-69.0.2-py3-none-any.whl (from https://pypi.org/simple/setuptools/) (requires-python:>=3.8), version: 69.0.2
    Found link https://files.pythonhosted.org/packages/4b/d9/d0cf66484b7e28a9c42db7e3929caed46f8b80478cd8c9bd38b7be059150/setuptools-69.0.2.tar.gz (from https://pypi.org/simple/setuptools/) (requires-python:>=3.8), version: 69.0.2
    F

In [ ]:
models_name_or_path="TheBloke/Llama-2-13B-chat-GGML"
model_basename="llama-2-13b-chat.ggmlv3.q5_1.bin"#Thge models is in bin format

#Import ALL the Required Libraries

In [ ]:
from huggingface_hub import hf_hub_download

In [ ]:
from llama_cpp import Llama

#Downlaod the Model

In [ ]:
print(models_name_or_path)
print(model_basename)
model_path=hf_hub_download(repo_id=models_name_or_path,filename=model_basename)

TheBloke/Llama-2-13B-chat-GGML
llama-2-13b-chat.ggmlv3.q5_1.bin


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


llama-2-13b-chat.ggmlv3.q5_1.bin: reconstructing file:   0%|          |  0.00B / 9.76GB            

llama-2-13b-chat.ggmlv3.q5_1.bin: downloading bytes:           |  0.00B            

In [ ]:
model_path


'/root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGML/snapshots/3140827b4dfcb6b562cd87ee3d7f07109b014dd0/llama-2-13b-chat.ggmlv3.q5_1.bin'

#Loading the Model

In [ ]:
#GPU
lcpp_llm=None
lcpp_llm=Llama(model_path=model_path,
               n_threads=2,#CPU cores
               n_batch=512,#Should be between 1 and n_ctx, consider the amount of VRAM in your GPU
               n_gpu_layers=32#Change this value based on yourm model and your GPU VRAM pool.
               )

AVX = 1 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | VSX = 0 | 


In [ ]:
#See the number of layers in GPU
lcpp_llm.params.n_gpu_layers

32

#Create a Prompt Template

In [ ]:
prompt="Write a MNIST image classification code in keras"
prompt_template=f'''System:Youare a helpful, respectful and honest assistant. Always answer as helpfully.

USER:{prompt}

ASSISTANT:
'''

#GENERATING A PROMPT TEMPLATE

In [ ]:
response=lcpp_llm(prompt=prompt_template,max_tokens=256,temperature=0.5,top_p=0.95,repeat_penalty=1.2,top_k=150,echo=True)

In [ ]:
print(response)

{'id': 'cmpl-6aa5ba0c-eb76-4300-abb6-714afba3ca17', 'object': 'text_completion', 'created': 1788257323, 'model': '/root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGML/snapshots/3140827b4dfcb6b562cd87ee3d7f07109b014dd0/llama-2-13b-chat.ggmlv3.q5_1.bin', 'choices': [{'text': 'System:Youare a helpful, respectful and honest assistant. Always answer as helpfully.\n\nUSER:Write a MNIST image classification code in keras\n\nASSISTANT:\n\nCertainly! Here is an example of how you could classify images using the MNIST dataset with Keras:\n```\n# Import necessary libraries\nfrom keras.models import Sequential\nfrom keras.layers import Dense, Dropout\nfrom keras.optimizers import Adam\nfrom keras.utils import to_categorical\nfrom keras.preprocessing.image import ImageDataGenerator\nimport numpy as np\n\n# Load the MNIST dataset\n(X_train, y_train), (X_test, y_test) = mnist.load_data()\n\n# Preprocess the images by resizing them to 28x28 pixels and flattening them\nX_train = X_train

In [ ]:
print(response['choices'][0]['text'])

System:Youare a helpful, respectful and honest assistant. Always answer as helpfully.

USER:Write a MNIST image classification code in keras

ASSISTANT:

Certainly! Here is an example of how you could classify images using the MNIST dataset with Keras:
```
# Import necessary libraries
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.utils import to_categorical
from keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Load the MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Preprocess the images by resizing them to 28x28 pixels and flattening them
X_train = X_train.reshape(-1, 28, 28, 1) / 255
X_test = X_test.reshape(-1, 28, 28, 1) / 255

# Convert the labels to categorical values
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)


